In [ ]:
!pip install openenv-core unsloth trl transformers wandb datasets


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)


In [ ]:
import random
import wandb

from models import SocraticAction
from server.environment import SocraticEnvironment
from students.scenarios import TRAINING_SCENARIOS

wandb.init(project="socratic-rl-training")

def socratic_reward(completions, **kwargs):
    rewards = []
    successes = 0
    for i, completion in enumerate(completions):
        env = SocraticEnvironment(seed=1000 + i)
        scenario = TRAINING_SCENARIOS[i % len(TRAINING_SCENARIOS)]
        env._scenario = scenario
        env._history = []
        env._understanding = 0.0
        env._turn = 0
        total_reward = 0.0
        turns = [t.strip() for t in str(completion).split("[TURN]") if t.strip()]
        for turn_text in turns[:5]:
            question = turn_text.splitlines()[0].strip()
            obs = env.step(SocraticAction(question=question))
            total_reward += float(obs.reward)
            if obs.done:
                break
        if env._understanding >= 0.90:
            successes += 1
        rewards.append(total_reward)
    mean_reward = sum(rewards) / max(len(rewards), 1)
    success_rate = successes / max(len(rewards), 1)
    wandb.log({"mean_reward": mean_reward, "success_rate": success_rate})
    return rewards


In [ ]:
from datasets import Dataset
from students.scenarios import TRAINING_SCENARIOS

rows = []
for _ in range(50):
    for s in TRAINING_SCENARIOS:
        prompt = (
            "You are a Socratic tutor.\\n"
            f"Topic: {s.topic}\\n"
            f"Student misconception: {s.misconception}\\n"
            "Rules:\\n"
            "1) Ask questions only.\\n"
            "2) Never state the answer directly.\\n"
            "3) Build on student responses turn by turn.\\n"
            "Respond with one strong Socratic question."
        )
        rows.append({"prompt": prompt})

dataset = Dataset.from_list(rows)
dataset


In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./socratic-agent",
    num_generations=4,
    max_completion_length=256,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    report_to="wandb",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[socratic_reward],
    args=training_args,
    train_dataset=dataset,
)

trainer.train()


In [ ]:
model.save_pretrained("socratic-agent-final")
model.push_to_hub("YOUR_USERNAME/socratic-rl-agent")
print("Training done. Now run: python eval.py --model_path ./socratic-agent-final")
